# Claude 3 vs Claude 4: migración y diferencias

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-4/03-comparativa-versiones.ipynb)

Script de migración automática y comparativa de comportamiento entre Claude 3 y Claude 4.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
client = anthropic.Anthropic()

# Tabla de migración
MIGRACION = {
    'claude-3-opus-20240229': 'claude-opus-4-7',
    'claude-3-5-sonnet-20241022': 'claude-sonnet-4-6',
    'claude-3-sonnet-20240229': 'claude-sonnet-4-6',
    'claude-3-haiku-20240307': 'claude-haiku-4-5-20251001',
    'claude-3-5-haiku-20241022': 'claude-haiku-4-5-20251001',
}

print('Tabla de migración Claude 3 → Claude 4:')
print(f"{'Modelo antiguo':<35} {'Modelo nuevo'}")
print('-' * 75)
for viejo, nuevo in MIGRACION.items():
    print(f"{viejo:<35} → {nuevo}")

## Script de migración automática de código

In [ ]:
import re

def migrar_codigo(codigo: str) -> tuple[str, int]:
    """Reemplaza IDs de modelo en código Python. Devuelve (código_nuevo, num_cambios)."""
    cambios = 0
    for viejo, nuevo in MIGRACION.items():
        if viejo in codigo:
            codigo = codigo.replace(viejo, nuevo)
            cambios += 1
    return codigo, cambios

# Ejemplo de código antiguo
codigo_antiguo = '''
import anthropic
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hola"}],
)
'''

codigo_nuevo, n_cambios = migrar_codigo(codigo_antiguo)
print(f'Cambios realizados: {n_cambios}')
print('Código migrado:')
print(codigo_nuevo)

## Comparativa de comportamiento: structured output

In [ ]:
import json

prompt_json = """Analiza este texto y devuelve SOLO un JSON con las claves:
- "sentimiento": "positivo", "negativo" o "neutro"
- "puntuacion": número del 1 al 10
- "palabras_clave": array de 3 palabras

Texto: 'El producto es increíble, la entrega fue rapidísima y el servicio al cliente excelente.'"""

# Claude 4 es más consistente devolviendo JSON puro
response = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=256,
    messages=[{'role': 'user', 'content': prompt_json}],
)

texto = response.content[0].text
print('Respuesta raw:', texto)

try:
    datos = json.loads(texto)
    print('\n✅ JSON válido parseado correctamente:')
    print(json.dumps(datos, ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print('⚠️ No es JSON puro - extraer con regex')

## Checklist de migración

```
✅ pip install anthropic>=0.49
✅ Reemplazar IDs de modelo con el script anterior
✅ Re-ejecutar suite de tests
✅ Verificar prompts de sistema (Claude 4 es más estricto)
✅ Revisar max_tokens (Claude 4 puede ser más verboso)
✅ Si usas Extended Thinking, añadir parámetro betas
```